# 🚀 Text-to-SQL Fine-tuning — Google Colab
## Qwen2.5-7B-Instruct × Unsloth × QLoRA

**Workflow tổng quát:**
1. Check GPU / CUDA
2. Cài dependencies (version-pinned, an toàn với Colab)
3. Mount Google Drive (lưu checkpoint & model)
4. Clone / upload source code
5. Override config cho Colab
6. (Tuỳ chọn) Resume từ checkpoint
7. Chạy training
8. Verify output

> ⚠️ **Yêu cầu:** Colab Pro với GPU A100 hoặc L4.  
> T4 (free tier) có thể bị OOM với Qwen2.5-7B dù dùng 4-bit.

---

## Cell 1 — Check GPU & CUDA

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA OK   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU       : {props.name}')
    print(f'VRAM      : {props.total_memory / 1024**3:.1f} GB')
else:
    raise RuntimeError('No GPU detected! Switch to GPU runtime: Runtime -> Change runtime type -> GPU')

## Cell 2 — Install Dependencies

**Version strategy:**
- `unsloth[colab-new]` từ git HEAD: tự detect CUDA/torch của Colab, tự cài xformers phù hợp
- `trl>=0.8.0,<0.10.0`: tránh breaking API của `SFTTrainer` ở trl 0.10+
- `transformers>=4.38.0,<4.50.0`: đủ mới cho Qwen2.5, không conflict Colab
- `bitsandbytes>=0.43.0`: cần cho `adamw_8bit` optimizer và 4-bit load
- ❌ **Không cài** `xformers` riêng (unsloth xử lý)
- ❌ **Không cài** `pyodbc` (Windows-only, không dùng được trên Colab Linux)

In [ ]:
# Bước 1: Cài unsloth trước — nó sẽ tự resolve xformers + triton tương thích
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Bước 2: Cài phần còn lại
!pip install \
    "transformers>=4.38.0,<4.50.0" \
    "trl>=0.8.0,<0.10.0" \
    "peft>=0.10.0,<0.13.0" \
    "accelerate>=0.28.0" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.18.0" \
    "sqlglot>=23.0.0" \
    "pyyaml" \
    -q

print('Dependencies installed successfully!')

In [ ]:
# Verify versions
import importlib
pkgs = ['unsloth', 'transformers', 'trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'sqlglot']
print('Package versions:')
for p in pkgs:
    try:
        m = importlib.import_module(p)
        print(f'  OK  {p:<20} {getattr(m, "__version__", "?")}') 
    except ImportError:
        print(f'  MISSING  {p}')

## Cell 3 — Mount Google Drive

Checkpoint và final model sẽ được lưu vào Drive để **không mất khi Colab session reset**.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ===========================================================================
# CONFIG: Thay đổi đường dẫn này nếu muốn lưu ở folder khác trên Drive
DRIVE_ROOT = '/content/drive/MyDrive/text-to-sql-finetune'
# ===========================================================================

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## Cell 4 — Setup Source Code

Chọn **một trong hai cách** bên dưới:

| Cách | Mô tả | Phù hợp khi |
|------|--------|-------------|
| **A** | Clone từ GitHub | Repo public/private trên GitHub |
| **B** | Copy từ Google Drive | Đã zip + upload source lên Drive |

> Sau khi setup xong, xác nhận biến `PROJECT_DIR` trỏ đúng vào thư mục chứa `src/`, `configs/`, `data/`.

In [ ]:
import os, sys

# ============================================================
# CHỌN CÁCH A: Clone từ GitHub
# ============================================================
# Uncomment và điền thông tin nếu dùng cách này

# from google.colab import userdata
# GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # lưu trong Colab Secrets
# REPO = 'YOUR_USERNAME/text-to-sql-finetune'
# !git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git /content/text-to-sql-finetune
# PROJECT_DIR = '/content/text-to-sql-finetune'


# ============================================================
# CHỌN CÁCH B: Copy từ Google Drive
# ============================================================
# Yêu cầu: đã upload thư mục project lên Drive trước
# (hoặc zip rồi unzip)

# !cp -r "{DRIVE_ROOT}/src"     /content/
# !cp -r "{DRIVE_ROOT}/configs" /content/
# !cp -r "{DRIVE_ROOT}/data"    /content/
# !cp -r "{DRIVE_ROOT}/prompts" /content/
# PROJECT_DIR = '/content'


# ============================================================
# Sau khi setup xong: set PROJECT_DIR
# ============================================================
PROJECT_DIR = '/content/text-to-sql-finetune'  # <-- điều chỉnh nếu dùng Cách B

os.chdir(PROJECT_DIR)
# Thêm project root vào sys.path để src.* imports hoạt động
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'CWD       : {os.getcwd()}')
print(f'sys.path  : {sys.path[:3]}')
print(f'Contents  : {os.listdir(".")}')

## Cell 5 — Cấu hình Training

Đọc `training_config.yaml` rồi override các giá trị phù hợp với Colab:
- `output_dir` → trỏ về Google Drive
- `save_checkpoints` → số epoch mỗi lần lưu (set `None` để tắt)
- batch size nhỏ hơn để tránh OOM

In [ ]:
import yaml, os

CONFIG_PATH = os.path.join(PROJECT_DIR, 'configs', 'training_config.yaml')
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# ============================================================
# OVERRIDE — chỉnh sửa theo nhu cầu
# ============================================================

# Output dir trên Drive (tránh mất khi session reset)
config['training']['output_dir'] = f'{DRIVE_ROOT}/outputs/qwen2.5-7b-text2sql'

# save_checkpoints: lưu LoRA checkpoint sau mỗi N epoch
# Set None để tắt. Với Colab, khuyến nghị = 1 (lưu sau mỗi epoch)
config['training']['save_checkpoints'] = 1

# Giảm batch size để phù hợp VRAM Colab (effective batch = 2x8 = 16)
config['training']['per_device_train_batch_size'] = 2
config['training']['gradient_accumulation_steps'] = 8

# Số epoch training
config['training']['num_train_epochs'] = 3

# Data paths (dùng absolute path)
config['data']['train_path'] = os.path.join(PROJECT_DIR, 'data', 'train.jsonl')
config['data']['val_path']   = os.path.join(PROJECT_DIR, 'data', 'val.jsonl')

# ============================================================

# Lưu runtime config ra file tạm để tiện debug
RUNTIME_CONFIG = '/content/runtime_config.yaml'
os.makedirs(config['training']['output_dir'], exist_ok=True)
with open(RUNTIME_CONFIG, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

effective_batch = config['training']['per_device_train_batch_size'] * config['training']['gradient_accumulation_steps']
print('Training config summary:')
print(f"  model            : {config['model']['name_or_path']}")
print(f"  load_in_4bit     : {config['model']['load_in_4bit']}")
print(f"  num_epochs       : {config['training']['num_train_epochs']}")
print(f"  batch (eff.)     : {effective_batch}")
print(f"  save_checkpoints : every {config['training']['save_checkpoints']} epoch(s)")
print(f"  output_dir       : {config['training']['output_dir']}")
print(f"  Runtime config   : {RUNTIME_CONFIG}")

## Cell 6 — (Tuỳ chọn) Resume từ Checkpoint

Nếu session Colab bị ngắt và bạn đã có checkpoint trên Drive,  
bỏ comment dòng `RESUME_FROM_CHECKPOINT = epoch_ckpts[-1]` bên dưới.

In [ ]:
import glob

OUTPUT_DIR = config['training']['output_dir']

# Tìm tất cả epoch checkpoints do EpochCheckpointCallback tạo ra
epoch_ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-epoch-*')))

# Tìm step checkpoints do HuggingFace Trainer tạo ra (save_strategy="steps")
step_ckpts = sorted([
    c for c in glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*'))
    if 'epoch' not in os.path.basename(c)
])

print('Epoch checkpoints (EpochCheckpointCallback):')
print(epoch_ckpts if epoch_ckpts else '  (none found)')

print('\nStep checkpoints (HF Trainer):')
print(step_ckpts if step_ckpts else '  (none found)')

# ============================================================
# SET RESUME CHECKPOINT
# None = train từ đầu
# Uncomment dòng dưới để resume từ checkpoint epoch mới nhất:
# RESUME_FROM_CHECKPOINT = epoch_ckpts[-1] if epoch_ckpts else None
# ============================================================
RESUME_FROM_CHECKPOINT = None

print(f'\nResume from: {RESUME_FROM_CHECKPOINT}')

## Cell 7 — Chạy Training

> Trainer sẽ tự động lưu checkpoint LoRA sau mỗi N epoch (theo `save_checkpoints` config)  
> vào Google Drive. Khi Colab bị ngắt, quay lại Cell 6 và set `RESUME_FROM_CHECKPOINT`.

In [ ]:
import os, sys, yaml

# Đảm bảo project root trong sys.path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments
from src.helpers import EpochCheckpointCallback
from src.training.dataset_loader import prepare_datasets

with open(RUNTIME_CONFIG, 'r') as f:
    config = yaml.safe_load(f)

# ── 1. Load model ────────────────────────────────────────────────────────────
# Nếu resume từ epoch checkpoint: load thẳng LoRA weights đã lưu
_model_source = RESUME_FROM_CHECKPOINT if RESUME_FROM_CHECKPOINT else config['model']['name_or_path']
print(f'Loading model from: {_model_source}')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=_model_source,
    max_seq_length=config['model']['max_seq_length'],
    dtype=None,           # auto: bfloat16 trên A100, float16 trên T4
    load_in_4bit=config['model']['load_in_4bit'],
)

# ChatML template cho Qwen2.5
tokenizer.chat_template = (
    "{%- for message in messages %}"
    "{%- if loop.first and messages[0]['role'] != 'system' %}"
    "{{ '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n' }}"
    "{%- endif %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] }}"
    "{%- if not loop.last %}{{ '<|im_end|>\n' }}{%- endif %}"
    "{%- endfor %}"
    "{%- if add_generation_prompt and messages[-1]['role'] != 'assistant' %}"
    "{{ '<|im_end|>\n<|im_start|>assistant\n' }}"
    "{%- endif %}"
)

# ── 2. LoRA PEFT ─────────────────────────────────────────────────────────────
# Khi resume từ epoch checkpoint, model đã có LoRA weights
# -> Chỉ get_peft_model khi train từ đầu
if not RESUME_FROM_CHECKPOINT:
    print('Setting up LoRA adapters...')
    model = FastLanguageModel.get_peft_model(
        model,
        r=config['lora']['r'],
        target_modules=config['lora']['target_modules'],
        lora_alpha=config['lora']['lora_alpha'],
        lora_dropout=config['lora']['lora_dropout'],
        bias=config['lora']['bias'],
        use_gradient_checkpointing='unsloth',
        random_state=config['training']['seed'],
    )
else:
    print('LoRA weights loaded from checkpoint, skipping get_peft_model()')

# ── 3. Datasets ───────────────────────────────────────────────────────────────
print('Loading datasets...')
train_ds, val_ds = prepare_datasets(
    train_path=config['data']['train_path'],
    val_path=config['data']['val_path'],
    tokenizer=tokenizer,
)
print(f'  Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

# ── 4. TrainingArguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    per_device_train_batch_size=config['training']['per_device_train_batch_size'],
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
    warmup_ratio=config['training']['warmup_ratio'],
    num_train_epochs=config['training']['num_train_epochs'],
    learning_rate=config['training']['learning_rate'],
    lr_scheduler_type=config['training']['lr_scheduler_type'],
    optim=config['training']['optim'],
    weight_decay=config['training']['weight_decay'],
    logging_steps=config['training']['logging_steps'],
    save_strategy=config['training']['save_strategy'],
    save_steps=config['training']['save_steps'],
    evaluation_strategy=config['training']['evaluation_strategy'],
    eval_steps=config['training']['eval_steps'],
    load_best_model_at_end=config['training']['load_best_model_at_end'],
    metric_for_best_model=config['training']['metric_for_best_model'],
    seed=config['training']['seed'],
    output_dir=config['training']['output_dir'],
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    report_to='none',
)

# ── 5. EpochCheckpointCallback ────────────────────────────────────────────────
callbacks = []
save_checkpoints = config['training'].get('save_checkpoints', None)
if save_checkpoints is not None:
    print(f'Epoch checkpoint enabled: saving every {save_checkpoints} epoch(s) to Drive')
    callbacks.append(EpochCheckpointCallback(
        model=model,
        tokenizer=tokenizer,
        output_dir=config['training']['output_dir'],
        save_every_n_epochs=int(save_checkpoints),
    ))

# ── 6. Trainer ────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=config['model']['max_seq_length'],
    dataset_num_proc=2,   # Colab thường không có nhiều CPU cores, dùng 2
    packing=False,
    args=training_args,
    callbacks=callbacks or None,
)

print('Starting training...')
trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

# ── 7. Save final model ───────────────────────────────────────────────────────
print('Saving final LoRA model to Drive...')
final_path = os.path.join(config['training']['output_dir'], 'final_lora')
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f'Done! Model saved to: {final_path}')

## Cell 8 — Verify Output

In [ ]:
import glob, os

OUTPUT_DIR = config['training']['output_dir']
print(f'Output dir: {OUTPUT_DIR}\n')

# Epoch checkpoints
epoch_ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-epoch-*')))
print('Epoch checkpoints saved:')
for c in epoch_ckpts:
    files = os.listdir(c)
    size  = sum(os.path.getsize(os.path.join(c, f)) for f in files
                if os.path.isfile(os.path.join(c, f)))
    print(f'  {os.path.basename(c):<35}  {size/1e6:.1f} MB  files={files}')

# Final model
final_path = os.path.join(OUTPUT_DIR, 'final_lora')
print('\nFinal model:')
if os.path.exists(final_path):
    print(f'  {os.listdir(final_path)}')
else:
    print('  Not saved yet')

---
## 📌 Hướng dẫn Resume khi Colab bị ngắt

### Kịch bản: GPU quota hết giữa epoch

1. **Chạy lại từ Cell 1** đến Cell 5 (check GPU, install, mount Drive, source, config)
2. Tại **Cell 6**, bỏ comment dòng:
   ```python
   RESUME_FROM_CHECKPOINT = epoch_ckpts[-1] if epoch_ckpts else None
   ```
3. Chạy **Cell 7** — Trainer sẽ:
   - Load LoRA weights từ checkpoint Drive
   - **Bỏ qua** `get_peft_model()` (đã có adapters)
   - Tiếp tục training từ epoch đã save

### Mất tối đa bao nhiêu progress?

| `save_checkpoints` | Mất tối đa |
|---|---|
| `1` | 1 epoch |
| `2` | 2 epoch |
| `None` | Toàn bộ (không save) |

> 💡 Khuyến nghị: đặt `save_checkpoints = 1` trên Colab free/pro để an toàn nhất.